# 7. PAR Estimation via FLoC-Based Yule-Walker


## Load the derived residual series

This notebook uses the residual series created by `02_decomposition.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

residual_series = pd.read_csv(
    "data/residual_series.csv",
    parse_dates=["date"],
    index_col="date"
)["residual"].dropna()

print(f"Loaded {len(residual_series)} residual observations.")


## Concepts

This notebook implements the FLoC-based Yule-Walker PAR estimation from the original analysis.

### Analogy
If ordinary relationship measures can be strongly affected by unusually large observations, an alternative robust relationship measure can provide a different estimation route. The exact estimator remains the implementation in the original notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Input residuals ===
y = residual_series.values
index = residual_series.index

# === Robust quantile estimator ===
def floc(x, y, tau=0.5):
    """
    FLoC estimator: ratio of quantiles of product vs square.
    Returns slope estimate for regressing y on x using quantile ratio.
    """
    numerator = np.quantile(y * x, tau)
    denominator = np.quantile(x ** 2, tau) + 1e-8  # to avoid division by zero
    return numerator / denominator

# === Fit AR(L) using FLoC instead of Yule-Walker ===
def floc_ar_coeffs(series, L, tau=0.5):
    coeffs = []
    for lag in range(1, L + 1):
        x = series[:-lag]
        y = series[lag:]
        coeff = floc(x, y, tau)
        coeffs.append(coeff)
    return np.array(coeffs)

# === Fit PAR(p, L) using FLoC dependencies ===
def fit_par_floc(y, p, L, tau=0.5):
    n = len(y)
    phi = {}
    fitted = np.zeros_like(y)
    
    for s in range(p):
        # Phase series: all y_t where t % p == s
        y_s = np.array([y[t] for t in range(p * L, n) if t % p == s])
        if len(y_s) <= L + 1:
            phi[s] = np.zeros(L)
            continue
        phi[s] = floc_ar_coeffs(y_s, L, tau)

    for t in range(p * L, n):
        s = t % p
        lag_vec = y[t - L:t][::-1]
        fitted[t] = np.dot(phi.get(s, np.zeros(L)), lag_vec)

    return phi, fitted

# === Compute AIC for FLoC-PAR ===
def compute_par_floc_aic(y, p, L):
    phi, fitted = fit_par_floc(y, p, L)
    residuals = y[p * L:] - fitted[p * L:]
    rss = np.sum(residuals ** 2)
    T = len(residuals)
    if T == 0 or rss == 0:
        return np.inf
    aic = T * np.log(rss / T) + 2 * (p * L)
    return aic

# === Grid search best (p, L) using AIC ===
best_aic = np.inf
best_p = None
best_L = None

for p in range(2, 31, 2):  # periods: 2, 4, ..., 30
    for L in range(1, 4):  # orders: 1, 2, 3
        aic = compute_par_floc_aic(y, p, L)
        if aic < best_aic:
            best_aic = aic
            best_p = p
            best_L = L

# === Final Fit using best (p, L) ===
phi, fitted = fit_par_floc(y, best_p, best_L)

# === Plot Actual vs Predicted ===
plt.figure(figsize=(12, 4))
plt.plot(index, y, label='Actual Residuals', alpha=0.6)
plt.plot(index, fitted, label=f'FLoC PAR({best_L}) Fit (p={best_p})', color='darkorange')
plt.title('PAR Model Fit using FLoC (Filtered Location Covariation)')
plt.xlabel('Time')
plt.ylabel('Residual Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# === Report Best Parameters ===
print(f"✅ Best Period (p): {best_p}")
print(f"✅ Best Order (L): {best_L}")
print(f"✅ Minimum AIC (FLoC): {best_aic:.2f}")
